In [28]:
import pandas as pd
import numpy as np


In [43]:
df_data = pd.read_csv("measured.csv") 

In [44]:
len(df_data)

272176

In [29]:
df_data = pd.read_csv("measured.csv") 
df_shots = pd.read_csv("shots.csv")   

df = df_data.merge(
    df_shots[["shot_id", "disrupted", "t_trigger_s"]], 
    on="shot_id", 
    how="left"
)

LEAD_TIME = 0.1 

df["y"] = (
    (df["disrupted"] == True) & 
    (df["t"] >= (df["t_trigger_s"] - LEAD_TIME)) & 
    (df["t"] <= df["t_trigger_s"])
).astype(int)

C:\Users\Gamer\AppData\Local\Temp\ipykernel_21796\2343930474.py:12: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["y"] = (


In [30]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier

In [31]:
def prepare_labeled_dataset(measured_path: str, shots_path: str, lead_time: float = 0.3) -> pd.DataFrame:
    """
    1. Загружает measured.csv и shots.csv
    2. Определяет момент термического гашения t_Q для сорвавшихся разрядов
    3. Создает таргет target_y = 1, если до срыва осталось <= lead_time секунд
    """
    df_meas = pd.read_csv(measured_path)
    df_shots = pd.read_csv(shots_path)

    disrupted_shots = set(df_shots[df_shots["disrupted"] == True]["shot_id"])

    t_q_map = {}
    for shot_id, group in df_meas.groupby("shot_id"):
        if shot_id in disrupted_shots:
            t_q_map[shot_id] = group["t"].max()
        else:
            t_q_map[shot_id] = np.inf

    df_meas["t_Q"] = df_meas["shot_id"].map(t_q_map)

    dt = df_meas["t_Q"] - df_meas["t"]
    df_meas["target_y"] = ((dt > 0) & (dt <= lead_time)).astype(int)

    df_meas = df_meas.drop(columns=["t_Q"])

    return df_meas

In [32]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler

In [ ]:

class LagFeatureExtractor(BaseEstimator, TransformerMixin):
  """Добавление лаговых признаков (истории за прошлые шаги)

  с корректным заполнением пропусков в начале разряда.
  """

  def __init__(self, shot_col='shot_id', lag_steps=[1, 3, 5]):
    self.shot_col = shot_col
    self.lag_steps = lag_steps

  def fit(self, X, y=None):
    return self

  def transform(self, X):
    X_out = X.copy()
    cols_to_lag = [
        'mag_ip_MA',
        'mag_saddle_lm',
        'ifr_chord_core',
        'ece_08',
        'bolo_01',
    ]
    available_cols = [c for c in cols_to_lag if c in X_out.columns]

    for col in available_cols:
      for lag in self.lag_steps:
        feat_name = f'{col}_lag{lag}'

        X_out[feat_name] = X_out.groupby(self.shot_col)[col].shift(lag)

        X_out[feat_name] = (
            X_out.groupby(self.shot_col)[feat_name].bfill().fillna(0.0)
        )

    return X_out

class ShotForwardFillImputer(BaseEstimator, TransformerMixin):
    """Шаг 1: Заполнение пропусков (NaN -> ffill) внутри каждого shot_id"""
    def __init__(self, shot_col='shot_id'):
        self.shot_col = shot_col

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        feature_cols = [c for c in X_out.columns if c not in [self.shot_col, 't']]
        X_out[feature_cols] = X_out.groupby(self.shot_col)[feature_cols].ffill()
        X_out[feature_cols] = X_out[feature_cols].fillna(0.0)
        return X_out


class TokamakPhysicsExtractor(BaseEstimator, TransformerMixin):
    """Шаг 2: Расчет физических соотношений (прокси Гринвальда, q95, излучение)"""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        
        
        A_MINOR = 0.95
        GREENWALD_AREA = np.pi * (A_MINOR ** 2)
        CHORD_M = 2.0 * A_MINOR * 1.70

        ip = np.maximum(X_out["mag_ip_MA"].abs(), 1e-4)
        bt = X_out["mag_bt_T"]
        chord = X_out["ifr_chord_core"]

        
        X_out["feat_f_gw_proxy"] = (chord / CHORD_M) * GREENWALD_AREA / ip

        
        X_out["feat_q95_proxy"] = 5.0 * (A_MINOR ** 2) * bt / (2.89 * ip)

        
        bolo_cols = [c for c in X_out.columns if c.startswith("bolo_")]
        p_rad = X_out[bolo_cols].sum(axis=1) if bolo_cols else 0.0
        p_in = X_out["nbi_power_MW"] + X_out["ich_power_MW"] + X_out["ech_power_MW"] + 1e-3
        X_out["feat_rad_fraction"] = p_rad / p_in

        
        X_out["feat_lm_abs"] = X_out["mag_saddle_lm"].abs()

        return X_out


class TimeSeriesDynamicsExtractor(BaseEstimator, TransformerMixin):
    """Шаг 3: Скользящие средние (EMA), разности и зачистка служебных колонок"""
    def __init__(self, shot_col='shot_id', alpha=0.1):
        self.shot_col = shot_col
        self.alpha = alpha

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_out = X.copy()
        
        
        X_out["feat_lm_ema"] = X_out.groupby(self.shot_col)["feat_lm_abs"].transform(
            lambda x: x.ewm(alpha=self.alpha, adjust=False).mean()
        )

        
        if "ece_08" in X_out.columns:
            X_out["feat_ece_diff"] = X_out.groupby(self.shot_col)["ece_08"].diff().fillna(0.0)

        
        drop_cols = [self.shot_col, 't']
        X_out = X_out.drop(columns=[c for c in drop_cols if c in X_out.columns])
        
        return X_out


def build_pipeline():
  return Pipeline([
      ('imputer', ShotForwardFillImputer(shot_col='shot_id')),
      ('physics', TokamakPhysicsExtractor()),
      ('lags', LagFeatureExtractor(shot_col='shot_id', lag_steps=[1, 3, 5])),
      (
          'dynamics',
          TimeSeriesDynamicsExtractor(shot_col='shot_id', alpha=0.1),
      ),
      (
          'model',
          XGBClassifier(
              random_state=42,
              eval_metric='logloss',
              tree_method='hist',
              n_jobs=-1,
          ),
      ),
  ])

In [34]:
from sklearn.model_selection import GroupKFold, GridSearchCV


In [35]:
from lightgbm import LGBMClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier

In [45]:
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier


def run_grid_search(X, y, groups):
    pipeline = build_pipeline()

    
    param_grid = [{
    'model': [
        XGBClassifier(
            random_state=42,
            eval_metric='logloss',
            tree_method='hist',
            n_jobs=-1,
        )
    ],
    'dynamics__alpha': [0.1],
    'model__n_estimators': [100, 150],
    'model__learning_rate': [0.02, 0.03],  # Медленное и стабильное обучение
    'model__max_depth': [3],  # Жесткое ограничение глубины (не дает зубрить шум)
    'model__min_child_weight': [
        5,
        10,
    ],  # Минимальный вес листа (штраф за мелкие листья)
    'model__gamma': [0.5, 1.0],  # Штраф за разделение узла дерева
    'model__colsample_bytree': [
        0.5,
        0.7,
    ],  # Случайный выбор фичей на каждом дереве
    'model__reg_alpha': [5.0, 10.0],  # Тяжелая L1 регуляризация
    'model__reg_lambda': [5.0, 10.0],  # Тяжелая L2 регуляризация
    'model__scale_pos_weight': [2.0, 3.0],
}]

    gkf = GroupKFold(n_splits=4)

    
    scoring = {'roc_auc': 'roc_auc', 'log_loss': 'neg_log_loss'}

    grid_search = GridSearchCV(
        estimator=pipeline,
        param_grid=param_grid,
        cv=gkf,
        scoring=scoring,
        refit='roc_auc',
        return_train_score=True,  
        n_jobs=-1,
        verbose=1,
    )

    grid_search.fit(X, y, groups=groups)

    
    results = grid_search.cv_results_
    print('\n=== СРАВНЕНИЕ МОДЕЛЕЙ И ПРОВЕРКА НА ОВЕРФИТ ===')
    print(
        f"{'Model':<28} | {'Train AUC':<10} | {'Val AUC':<10} | {'Train Loss':<10} | {'Val Loss':<10}"
    )
    print('-' * 77)

    for i in range(len(results['params'])):
        model_name = results['params'][i]['model'].__class__.__name__
        tr_auc = results['mean_train_roc_auc'][i]
        val_auc = results['mean_test_roc_auc'][i]
        tr_loss = -results['mean_train_log_loss'][i]
        val_loss = -results['mean_test_log_loss'][i]

        print(
            f'{model_name:<28} | {tr_auc:.4f}     | {val_auc:.4f}     | {tr_loss:.4f}     | {val_loss:.4f}'
        )

    print(f"\nЛучшая модель по ROC-AUC: {grid_search.best_score_:.4f}")
    return grid_search.best_estimator_

In [ ]:
MEASURED_FILE = "measured.csv"
SHOTS_FILE = "shots.csv"

print("Подготовка данных...")
df = prepare_labeled_dataset(MEASURED_FILE, SHOTS_FILE, lead_time=0.3)

X = df.drop(columns=["target_y"])
y = df["target_y"]
groups = df["shot_id"]

best_pipeline = run_grid_search(X, y, groups)